# 골드셋 생성 (v2 — LLM 분절)

강의 전사를 윈도우 단위로 LLM에 분절시켜 **'개념/예시 + 그 구간'**을 추출한다.
초안은 `*_v2.xlsx` 의 `human_check` 로 사람이 검수해 확정한다.
(v1 키워드 방식·토픽 그룹핑은 폐기)


In [28]:
import asyncio
import json
import re
from pathlib import Path

import pandas as pd
import google.generativeai as genai
from tqdm.auto import tqdm

import sys
sys.path.insert(0, str(Path(".").resolve().parent))

from app.core.config import settings

genai.configure(api_key=settings.api_key)
MODEL = settings.eval_model
print("model:", MODEL)

model: models/gemini-2.5-pro


In [14]:
# ── 분석할 날짜 지정 ─────────────────────────────────────────────────
TARGET_DATE = "2026-02-09"
CSV_PATH = Path("../data/processed/lectures_kss.csv")

df_all = pd.read_csv(CSV_PATH)
df = df_all[df_all["date"] == TARGET_DATE].reset_index(drop=True)
sentences = df["text_raw"].tolist()
print(f"날짜: {TARGET_DATE}, 총 문장 수: {len(sentences)}")
df.head(3)

날짜: 2026-02-09, 총 문장 수: 1646


,lecture_id,date,timestamp,speaker_id,text_raw
0,kdt-backendj-21th,2026-02-09,09:10:24,e241d60d,여분 안녕하세요.
1,kdt-backendj-21th,2026-02-09,09:10:24,e241d60d,이번주 수업 진행하도록 하겠습니다.
2,kdt-backendj-21th,2026-02-09,09:10:37,e241d60d,우리가 이제 지난 시간에는 저희 문자 함수 날짜 시간 집계 윈도우 함수 그 다음에 ...


---

## 개념 분절 골드셋 (v2) — 개념 1개 = 1덩어리

기존 방식(개념 *문장 인덱스* 추출 → 위치 기반 병합)은 키워드 매칭 false positive로 과추출되고,
한 개념이 여러 청크로 파편화된다. v2는 LLM에게 **"구별되는 개념 + 그 설명 구간"을 직접 분절**시킨다.

- 윈도우 단위(120문장, 20문장 겹침)로 인덱스 드리프트 방지
- 출력: `concept_name` + `start/end` 구간 + `key_sentence` → 개념 1개 = 1덩어리
- **이 결과는 초안(draft)** — `*_v2.xlsx`의 `human_check` 열로 사람이 경계·라벨을 검수해야 골드셋으로 확정.
  (goldset은 평가 대상 anchored_labeled보다 더 믿을 만해야 하므로 사람 검수는 생략 불가)


In [132]:
# ── 개념 분절(segmentation) 실행 ─────────────────────────────────────
# 문장 인덱스 나열이 아니라, "구별되는 개념 + 시작/끝 구간"을 윈도우 단위로 받는다.
SEG_SYSTEM_PROMPT = """당신은 강의 전사 텍스트에서 '구별되는 개념'을 식별하고
각 개념이 설명되는 구간을 하나의 덩어리로 묶는 전문가입니다.
응답은 반드시 JSON만 출력하세요."""

SEG_USER_TEMPLATE = """아래는 강의 전사의 한 구간입니다. 각 줄은 [전역인덱스] 문장 형식입니다.

이 구간에서 강사가 명시적으로 정의하거나 설명하는 '구별되는 개념'을 식별하세요.

【묶기 규칙 — 가장 중요】
- 하나의 개념이 여러 문장에 걸쳐 설명되면, 그 전체를 하나의 구간(start_index~end_index)으로 묶습니다.
- 중간에 짧은 곁가지(예시 한두 마디 등)가 껴도 같은 개념 설명이 이어지면 하나로 봅니다.
- 서로 다른 개념은 각각 별도 항목으로 분리합니다. (개념 1개 = 항목 1개)

【제외】
- 예시 들기, 실습 지시("해보세요"), 진행 안내, 잡담은 개념이 아닙니다.
- 개념을 '정의·설명'하지 않고 단순 언급만 하는 문장도 제외합니다.

각 개념에 대해 아래를 채우세요:
- concept_name: 개념의 핵심 이름 (예: "CONCAT 함수", "PRIMARY KEY와 UNIQUE의 차이")
- start_index / end_index: 그 개념 설명의 시작/끝 문장의 전역 인덱스 (반드시 아래 구간 내 값)
- key_sentence: 그 개념을 가장 잘 정의·설명하는 핵심 문장 원문 1개

문장 구간:
{indexed_sentences}

아래 JSON 형식으로만 응답하세요:
{{"concepts": [{{"concept_name": "...", "start_index": 0, "end_index": 0, "key_sentence": "..."}}]}}"""

WINDOW = 120   # 한 번에 보여줄 문장 수 (인덱스 추적 안정 + 충분한 문맥)
STRIDE = 100   # 윈도우 간 이동 → 20문장 겹침(경계 걸친 개념 보존)


def _format_window(sentences: list[str], start: int, end: int) -> str:
    return "\n".join(f"[{i}] {sentences[i]}" for i in range(start, end))


def _parse_concepts(raw: str) -> list[dict]:
    text = re.sub(r"^```\w*\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m:
            return []
        data = json.loads(m.group())
    return data.get("concepts", [])


def run_segmentation(sentences: list[str], window: int = WINDOW, stride: int = STRIDE) -> list[dict]:
    model = genai.GenerativeModel(MODEL)
    out: list[dict] = []
    for ws in tqdm(range(0, len(sentences), stride), desc="segmentation"):
        we = min(ws + window, len(sentences))
        prompt = f"{SEG_SYSTEM_PROMPT}\n\n{SEG_USER_TEMPLATE.format(indexed_sentences=_format_window(sentences, ws, we))}"
        resp = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                temperature=settings.llm_temperature,
            ),
        )
        for c in _parse_concepts(resp.text):
            try:
                s, e = int(c["start_index"]), int(c["end_index"])
            except (KeyError, ValueError, TypeError):
                continue
            if s > e:
                s, e = e, s
            s, e = max(s, ws), min(e, we - 1)   # 윈도우 범위로 클램프
            if s > e:
                continue
            out.append({
                "concept_name": str(c.get("concept_name", "")).strip(),
                "start": s,
                "end": e,
                "key_sentence": str(c.get("key_sentence", "")).strip(),
            })
        if we >= len(sentences):
            break
    return out


def dedup_concepts(items: list[dict]) -> list[dict]:
    """겹치는 구간(윈도우 중복)을 같은 개념으로 보고 병합. 인접만 한 건 분리 유지."""
    merged: list[dict] = []
    for c in sorted(items, key=lambda x: (x["start"], x["end"])):
        if merged and c["start"] <= merged[-1]["end"]:        # 구간이 실제로 겹침 → 동일 개념
            prev = merged[-1]
            prev["end"] = max(prev["end"], c["end"])
            if len(c["concept_name"]) > len(prev["concept_name"]):  # 더 구체적인 이름 유지
                prev["concept_name"] = c["concept_name"]
        else:
            merged.append(dict(c))
    return merged


concepts_raw = run_segmentation(sentences)
concepts = dedup_concepts(concepts_raw)
for c in concepts:
    c["text"] = " ".join(sentences[i] for i in range(c["start"], c["end"] + 1))
    c["timestamp"] = df.loc[c["start"], "timestamp"]

print(f"분절 결과: {len(concepts_raw)}개 → dedup → {len(concepts)}개 개념")
print(f"(참고) 기존 키워드 방식 개념 청크: 121개")
print()
for c in concepts[:8]:
    print(f"[{c['start']}~{c['end']}] {c['concept_name']}")
    print(f"  key: {c['key_sentence'][:80]}")


segmentation:  94%|█████████▍| 16/17 [11:17<00:42, 42.32s/it]

분절 결과: 78개 → dedup → 63개 개념
(참고) 기존 키워드 방식 개념 청크: 121개

[26~29] SUBSTR/SUBSTRING 함수
  key: 데이터 변하는 걸 하는데 추출하고 가공하고 결과를 전송하 이런 ETA을 하는데 ETL 할 때 가장 많이 사용해서 손을 잡는 것이 서브SST야
[31~38] LENGTH 함수의 바이트 기준과 플랫폼 의존성
  key: 실제 글자가 한 글자가 바이트를 얼마큼 차지하는지는 OS 플랫폼에도 조금 영향을 받는다.
[40~41] LEFT/RIGHT 함수
  key: 이 레프트는 왼쪽에서 일단은 몇 글자를 추출할 건지예요.
[45~47] REPLACE 함수
  key: 그러면은 A만 찾아서 X를 해서 치환을 해서 전체 리턴하는데 특정 문자 치환이에요.
[48~49] TRIM 함수
  key: 트링 같은 경우에는 앞뒤 공백 제거를 해준다.
[50~54] INSTR과 LOCATE 함수의 차이
  key: R패드 같은 경우에는 전체 자릿수에 해당 필드 내용을 리턴을 해주고 채우는 거죠.
[68~71] CONCAT 함수의 NULL 처리
  key: 같은 경우에는 결합 문자를 그대로 나타내는 거고, 만약에 중간에 널 있으면 널로 리턴해준다.
[86~94] CONCAT_WS 함수의 NULL 처리
  key: 보니 세퍼레이터는 무조건 살려야 되는 세퍼레이터는 무조건 트루기 때문에 기본적으로 널이 온다 그러면 널은 무시해버린다


In [133]:
# ── v2 골드셋 저장 (사람 검수용) ─────────────────────────────────────
# JSON: 채점/정렬 파이프라인이 읽을 형식  /  XLSX: 사람이 경계·라벨 검수하는 시트
import openpyxl
from openpyxl.styles import Alignment

seg_goldset = {
    "source": CSV_PATH.name,
    "date": TARGET_DATE,
    "total_sentences": len(sentences),
    "method": "llm_segmentation_v2",
    "concept_count": len(concepts),
    "concepts": concepts,
}
seg_json_path = Path("../data/goldset") / f"{TARGET_DATE}_concept_goldset_v2.json"
seg_json_path.parent.mkdir(parents=True, exist_ok=True)
seg_json_path.write_text(json.dumps(seg_goldset, ensure_ascii=False, indent=2), encoding="utf-8")

# 검수 시트: human_check 열 표기 → O=개념맞음 / X=개념아님 / M=두 개념이 한 덩어리(나눠야) / S=한 개념이 쪼개짐(합쳐야)
rows = [
    {
        "timestamp": c["timestamp"],
        "concept_name": c["concept_name"],
        "span": f"{c['start']}~{c['end']}",
        "key_sentence": c["key_sentence"],
        "text": c["text"],
        "human_check": "",
    }
    for c in concepts
]
seg_xlsx_path = Path("../data/goldset") / f"{TARGET_DATE}_concept_goldset_v2.xlsx"
pd.DataFrame(rows).to_excel(seg_xlsx_path, index=False)

wb = openpyxl.load_workbook(seg_xlsx_path)
ws = wb.active
for col, w in {"A": 12, "B": 28, "C": 10, "D": 50, "E": 70, "F": 12}.items():
    ws.column_dimensions[col].width = w
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
wb.save(seg_xlsx_path)

print(f"JSON 저장: {seg_json_path}")
print(f"검수 시트 저장: {seg_xlsx_path} ({len(concepts)}개 개념)")
print("\n검수 표기법: O=개념맞음 / X=개념아님 / M=두 개념이 한 덩어리(나눠야) / S=한 개념이 쪼개짐(합쳐야)")


JSON 저장: ../data/goldset/2026-02-09_concept_goldset_v2.json
검수 시트 저장: ../data/goldset/2026-02-09_concept_goldset_v2.xlsx (63개 개념)

검수 표기법: O=개념맞음 / X=개념아님 / M=두 개념이 한 덩어리(나눠야) / S=한 개념이 쪼개짐(합쳐야)


---

## 예시 분절 골드셋 (v2) — 예시 1개 = 1덩어리

개념 v2와 동일한 방법론. LLM에게 **"구별되는 예시 + 그 구간"**을 윈도우 단위로 분절시킨다.
- 예시 = 구체적 값·이름·숫자로 함수/개념의 동작·결과를 *보여주는* 구간
- 출력: `example_name` + `start/end` + `key_sentence` → 예시 1개 = 1덩어리
- 초안 → `*_example_goldset_v2.xlsx` 의 `human_check` 로 사람 검수해 확정
- (개념 v2 셀을 먼저 실행해야 `WINDOW/STRIDE/_format_window` 가 정의됨)

In [137]:
# (자립 실행용) 개념 v2와 공유하는 윈도우 설정·헬퍼
WINDOW = WINDOW if 'WINDOW' in dir() else 120
STRIDE = STRIDE if 'STRIDE' in dir() else 100
if '_format_window' not in dir():
    def _format_window(sentences, start, end):
        return "\n".join(f"[{i}] {sentences[i]}" for i in range(start, end))

# ── 예시 분절(segmentation) 실행 ─────────────────────────────────────
EX_SEG_SYSTEM_PROMPT = """당신은 강의 전사 텍스트에서 '예시(example)'를 식별하고
각 예시가 제시되는 구간을 하나의 덩어리로 묶는 전문가입니다.
응답은 반드시 JSON만 출력하세요."""

EX_SEG_USER_TEMPLATE = """아래는 강의 전사의 한 구간입니다. 각 줄은 [전역인덱스] 문장 형식입니다.

이 구간에서 강사가 **구체적인 값·이름·숫자·상황을 들어 함수/개념의 동작이나 결과를
보여주는 '예시'** 를 식별하세요.

【포함】
- "예를 들어 ~", "만약에 X를 넣으면 Y가 된다" 처럼 구체 입력→결과를 보여주는 구간
- 특정 숫자·이름·문자로 동작을 시연하는 구간

【묶기 규칙 — 가장 중요】
- 하나의 예시가 여러 문장에 걸치면 그 전체를 하나의 구간(start_index~end_index)으로 묶습니다.
- 서로 다른 예시는 각각 별도 항목으로 분리합니다. (예시 1개 = 항목 1개)

【제외】
- 구체 값 없이 일반적으로 정의·설명만 하는 문장(= 개념)
- 수강생 독립 과제(= 실습), 진행 안내·잡담

각 예시에 대해:
- example_name: 무엇을 보여주는 예시인지 짧은 이름
- start_index / end_index: 시작/끝 문장의 전역 인덱스 (반드시 구간 내 값)
- key_sentence: 핵심 문장 원문 1개

문장 구간:
{indexed_sentences}

아래 JSON 형식으로만 응답하세요:
{{"examples": [{{"example_name": "...", "start_index": 0, "end_index": 0, "key_sentence": "..."}}]}}"""


def _parse_examples(raw: str) -> list[dict]:
    text = re.sub(r"^```\w*\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if not m:
            return []
        data = json.loads(m.group())
    return data.get("examples", [])


def run_example_segmentation(sentences, window=WINDOW, stride=STRIDE):
    model = genai.GenerativeModel(MODEL)
    out = []
    for ws in tqdm(range(0, len(sentences), stride), desc="example-seg"):
        we = min(ws + window, len(sentences))
        prompt = f"{EX_SEG_SYSTEM_PROMPT}\n\n{EX_SEG_USER_TEMPLATE.format(indexed_sentences=_format_window(sentences, ws, we))}"
        resp = model.generate_content(prompt, generation_config=genai.GenerationConfig(
            response_mime_type="application/json", temperature=settings.llm_temperature))
        for c in _parse_examples(resp.text):
            try:
                s, e = int(c["start_index"]), int(c["end_index"])
            except (KeyError, ValueError, TypeError):
                continue
            if s > e:
                s, e = e, s
            s, e = max(s, ws), min(e, we - 1)
            if s > e:
                continue
            out.append({"example_name": str(c.get("example_name", "")).strip(),
                        "start": s, "end": e,
                        "key_sentence": str(c.get("key_sentence", "")).strip()})
        if we >= len(sentences):
            break
    return out


def dedup_examples(items):
    merged = []
    for c in sorted(items, key=lambda x: (x["start"], x["end"])):
        if merged and c["start"] <= merged[-1]["end"]:
            merged[-1]["end"] = max(merged[-1]["end"], c["end"])
            if len(c["example_name"]) > len(merged[-1]["example_name"]):
                merged[-1]["example_name"] = c["example_name"]
        else:
            merged.append(dict(c))
    return merged


examples = dedup_examples(run_example_segmentation(sentences))
for c in examples:
    c["text"] = " ".join(sentences[i] for i in range(c["start"], c["end"] + 1))
    c["timestamp"] = df.loc[c["start"], "timestamp"]

print(f"예시 분절 결과: {len(examples)}개")
for c in examples[:8]:
    print(f"[{c['start']}~{c['end']}] {c['example_name']}")

example-seg:  94%|█████████▍| 16/17 [10:55<00:40, 40.97s/it]

예시 분절 결과: 82개
[43~43] RIGHT 함수 예시
[46~46] REPLACE 함수 예시
[72~75] CONCAT 함수로 숫자/문자열 결합
[79~81] CONCAT 함수와 NULL 처리
[99~103] CONCAT 함수 실전 예제 (이름과 직업 연결)
[104~104] CONCAT_WS 함수 실전 예제 (이름과 직업 연결)
[145~148] SUBSTR로 세 글자 추출 예시
[151~153] SUBSTR FROM POS 구문 사용 예시


In [138]:
# ── 예시 v2 골드셋 저장 (사람 검수용) ─────────────────────────────────
import openpyxl
from openpyxl.styles import Alignment

ex_goldset = {
    "source": CSV_PATH.name, "date": TARGET_DATE,
    "method": "llm_segmentation_v2", "example_count": len(examples), "examples": examples,
}
ex_json_path = Path("../data/goldset") / f"{TARGET_DATE}_example_goldset_v2.json"
ex_json_path.write_text(json.dumps(ex_goldset, ensure_ascii=False, indent=2), encoding="utf-8")

rows = [{"timestamp": c["timestamp"], "example_name": c["example_name"],
         "span": f"{c['start']}~{c['end']}", "key_sentence": c["key_sentence"],
         "text": c["text"], "human_check": ""} for c in examples]
ex_xlsx_path = Path("../data/goldset") / f"{TARGET_DATE}_example_goldset_v2.xlsx"
pd.DataFrame(rows).to_excel(ex_xlsx_path, index=False)
wb = openpyxl.load_workbook(ex_xlsx_path); ws = wb.active
for col, w in {"A": 12, "B": 30, "C": 12, "D": 45, "E": 70, "F": 12}.items():
    ws.column_dimensions[col].width = w
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
wb.save(ex_xlsx_path)
print(f"JSON 저장: {ex_json_path}")
print(f"검수 시트 저장: {ex_xlsx_path} ({len(examples)}개 예시)")
print("검수 표기: O=예시맞음 / X=예시아님 / M=나눠야 / S=합쳐야")

JSON 저장: ../data/goldset/2026-02-09_example_goldset_v2.json
검수 시트 저장: ../data/goldset/2026-02-09_example_goldset_v2.xlsx (82개 예시)
검수 표기: O=예시맞음 / X=예시아님 / M=나눠야 / S=합쳐야


---

### 예시 토픽 그룹핑 (검수 가속용)

예시 분절 결과(`examples`)를 '무엇을 보여주는 예시인가'로 묶어 토픽 시트를 만든다.
**토픽 단위로 O/X 를 빠르게 찍고**, 이후 O 토픽만 멤버(예시)로 펼쳐 확정한다(개념과 동일).
산출: `*_example_goldset_topic.xlsx` (토픽 1행).

In [139]:
# ── 예시 토픽 그룹핑 (LLM 1회, 검수 가속용) ──────────────────────────
import openpyxl
from openpyxl.styles import Alignment

EX_GROUP_PROMPT = """아래는 한 강의에서 추출한 예시 목록입니다 (인덱스: 예시명 (구간)).
'무엇을 보여주는 예시인가'(같은 함수/주제)끼리 묶어 토픽으로 그룹화하세요.

【규칙】
- 같은 함수/개념을 보여주는 예시끼리 한 토픽으로 (예: SUBSTR 예시 여러 개 → "SUBSTR 예시").
- 모든 예시는 반드시 어느 한 토픽에 속해야 합니다 (누락 금지).

예시 목록:
{items}

아래 JSON 형식으로만 응답하세요:
{{"topics": [{{"topic_name": "...", "member_indices": [0, 3]}}, ...]}}"""


def group_examples(examples):
    model = genai.GenerativeModel(MODEL)
    items = "\n".join(f"[{i}] {c['example_name']} ({c['start']}~{c['end']})" for i, c in enumerate(examples))
    resp = model.generate_content(EX_GROUP_PROMPT.format(items=items),
        generation_config=genai.GenerationConfig(response_mime_type="application/json", temperature=settings.llm_temperature))
    t = re.sub(r"^```\w*\s*", "", resp.text.strip())
    t = re.sub(r"\s*```$", "", t).strip()
    return json.loads(t).get("topics", [])


ex_topics_raw = group_examples(examples)
ex_topics, used = [], set()
for t in ex_topics_raw:
    idxs = [i for i in t.get("member_indices", []) if isinstance(i, int) and 0 <= i < len(examples) and i not in used]
    if not idxs:
        continue
    used.update(idxs)
    mem = sorted((examples[i] for i in idxs), key=lambda c: c["start"])
    ex_topics.append({"topic_name": str(t.get("topic_name", "")).strip(),
                      "member_names": [m["example_name"] for m in mem],
                      "spans": [[m["start"], m["end"]] for m in mem],
                      "key_sentences": [m["key_sentence"] for m in mem],
                      "start": mem[0]["start"]})
for i in range(len(examples)):
    if i not in used:
        c = examples[i]
        ex_topics.append({"topic_name": c["example_name"], "member_names": [c["example_name"]],
                          "spans": [[c["start"], c["end"]]], "key_sentences": [c["key_sentence"]],
                          "start": c["start"], "_ungrouped": True})
ex_topics.sort(key=lambda t: t["start"])

(Path("../data/goldset") / f"{TARGET_DATE}_example_goldset_topic.json").write_text(
    json.dumps({"date": TARGET_DATE, "topic_count": len(ex_topics), "topics": ex_topics},
               ensure_ascii=False, indent=2), encoding="utf-8")

rows = [{"timestamp": df.loc[t["start"], "timestamp"], "topic_name": t["topic_name"],
         "n_members": len(t["member_names"]), "spans": ", ".join(f"{s}~{e}" for s, e in t["spans"]),
         "members": " | ".join(t["member_names"]),
         "key_sentences": "\n".join(f"- {k}" for k in t["key_sentences"]),
         "ungrouped": "Y" if t.get("_ungrouped") else "", "human_check": ""} for t in ex_topics]
ex_topic_xlsx = Path("../data/goldset") / f"{TARGET_DATE}_example_goldset_topic.xlsx"
pd.DataFrame(rows).to_excel(ex_topic_xlsx, index=False)
wb = openpyxl.load_workbook(ex_topic_xlsx); ws = wb.active
for col, w in {"A": 12, "B": 26, "C": 10, "D": 22, "E": 45, "F": 70, "G": 10, "H": 12}.items():
    ws.column_dimensions[col].width = w
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical="top")
wb.save(ex_topic_xlsx)

print(f"예시 토픽 {len(ex_topics)}개 → 검수 시트: {ex_topic_xlsx}")
print("검수: 토픽 단위로 O/X 빠르게. 이후 O 토픽만 멤버로 펼쳐 확정.")

예시 토픽 20개 → 검수 시트: ../data/goldset/2026-02-09_example_goldset_topic.xlsx
검수: 토픽 단위로 O/X 빠르게. 이후 O 토픽만 멤버로 펼쳐 확정.
